In [20]:
# =======================
# 1. Imports
# =======================
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import joblib

# =======================
# 2. Custom Dataset
# =======================
class MultimodalDataset(Dataset):
    def __init__(self, clin_path, mrna_path, mut_path, labels_path):
        # Load preprocessed data
        self.clinical = joblib.load(clin_path).values.astype(float)
        self.mrna = joblib.load(mrna_path).values.astype(float)
        self.mutation = joblib.load(mut_path).values.astype(float)
        self.labels = joblib.load(labels_path).astype(float)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return (
            torch.tensor(self.clinical[idx], dtype=torch.float32),
            torch.tensor(self.mrna[idx], dtype=torch.float32),
            torch.tensor(self.mutation[idx], dtype=torch.float32),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

# =======================
# 3. Paths
# =======================
train_dataset = MultimodalDataset(
    "../data/train/clinical.pkl",
    "../data/train/mrna.pkl",
    "../data/train/mutation.pkl",
    "../data/train/labels.pkl"
)

val_dataset = MultimodalDataset(
    "../data/val/clinical.pkl",
    "../data/val/mrna.pkl",
    "../data/val/mutation.pkl",
    "../data/val/labels.pkl"
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =======================
# 4. Simple Neural Network
# =======================
class SimpleMultimodalNet(nn.Module):
    def __init__(self, clin_dim, mrna_dim, mut_dim, hidden_dim=64):
        super().__init__()
        # Separate encoders for each modality
        self.clinical_fc = nn.Sequential(nn.Linear(clin_dim, hidden_dim), nn.ReLU())
        self.mrna_fc = nn.Sequential(nn.Linear(mrna_dim, hidden_dim), nn.ReLU())
        self.mut_fc = nn.Sequential(nn.Linear(mut_dim, hidden_dim), nn.ReLU())
        
        # Fusion layer
        self.fusion_fc = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),  # Binary output
            nn.Sigmoid()
        )
        
    def forward(self, clin, mrna, mut):
        clin_emb = self.clinical_fc(clin)
        mrna_emb = self.mrna_fc(mrna)
        mut_emb = self.mut_fc(mut)
        
        # Concatenate embeddings
        fused = torch.cat([clin_emb, mrna_emb, mut_emb], dim=1)
        output = self.fusion_fc(fused)
        return output.squeeze()

# =======================
# 5. Initialize Model
# =======================
# Use one batch to get dimensions
sample_batch = next(iter(train_loader))
clin_dim = sample_batch[0].shape[1]
mrna_dim = sample_batch[1].shape[1]
mut_dim = sample_batch[2].shape[1]
model = SimpleMultimodalNet(clin_dim, mrna_dim, mut_dim)

# =======================
# 6. Loss and Optimizer
# =======================
criterion = nn.BCELoss()  # binary cross entropy
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# =======================
# 7. Minimal Training Loop
# =======================
for epoch in range(2):  # just 2 epochs to test
    model.train()
    for clin, mrna, mut, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(clin, mrna, mut)
        print("Outputs range:", outputs.min().item(), outputs.max().item())
        print("Labels unique:", labels.unique())
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} - Loss: {loss.item():.4f}")
    print(outputs.min(), outputs.max())
    print(labels.unique())



C:\Users\gench\AppData\Local\Temp\ipykernel_22688\1958386516.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  torch.tensor(self.labels[idx], dtype=torch.float32)


Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Epoch 1 - Loss: 32.1429
tensor(0., grad_fn=<MinBackward1>) tensor(0., grad_fn=<MaxBackward1>)
tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs range: 0.0 0.0
Labels unique: tensor([0., 1.])
Outputs r

In [21]:
# Check output range
sample_batch = next(iter(train_loader))
mrna, mut, clin, labels = sample_batch
with torch.no_grad():
    outputs = model(mrna, mut, clin)
print(outputs.min(), outputs.max())  # should be between 0 and 1 for BCELoss

tensor(0.) tensor(0.)


C:\Users\gench\AppData\Local\Temp\ipykernel_22688\1958386516.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  torch.tensor(self.labels[idx], dtype=torch.float32)
